
# 📈 The Effects of Phone Calls on Voter Turnout - CACE Analysis

## 🎯 Part 1: What is this project about?

This project investigates the **causal impact of canvassing phone calls** on **voter turnout** in the August 2008 Michigan primary election. The central research question is:

> *"Do phone calls encouraging voting increase actual voter turnout, and if so, what is the magnitude of the effect among those who comply?"*

### 🧪 Why CACE?

Not all individuals assigned to treatment (receiving a phone call) were actually contacted. This leads us to estimate the **Complier Average Causal Effect (CACE)**, which captures the treatment effect **only among those who complied with their assigned condition**.

### 🔧 Tools Used

- **Coding Tools**: Python, `pandas`, `statsmodels`, `pyfixest`, `matplotlib`, `seaborn`
- **Mathematical Tools**:
+ Intent-to-Treat (ITT) analysis to estimate the average effect of treatment assignment.

+ Compliance Rate (α) to determine the proportion of treated individuals who were actually contacted.

+ CACE calculation (Manual and IV) to estimate the treatment effect among compliers.

+ Instrumental Variable (IV) regression using pyfixest to confirm CACE estimates.

+ OLS regressions to inspect placebo and heterogeneity effects.
These tools help distinguish between correlation and causality, and accurately estimate the treatment effect despite noncompliance.



## 📊 Part 2: Data

This study, conducted in Michigan ahead of the August 2008 primary election, assigned voters to one of three conditions:

- **Control**: Received no phone call.
- **Placebo**: Received a phone call about recycling.
- **Treatment**: Received a phone call encouraging voting.

The outcome, `voted_aug2008`, is binary: **1 if voted**, **0 otherwise**.

### 🗃️ Data Files

- `noncompliance_treat_small.csv`: Contains **control and treatment** groups.
- `noncompliance_placebo.csv`: Contains **placebo and treatment** groups.

### 🧾 Variables

- `voted_aug2008`: Outcome variable — whether the person voted.
- `contacted`: Whether the person actually answered the call (compliance).
- `treatment_attempt_turnout_call`: Whether the person was assigned to receive a turnout call.
- `voted_nov2002`: Whether the person voted in the 2002 general election (covariate).


In [6]:
import pandas as pd

# Load both datasets
from google.colab import drive
drive.mount('/content/drive')
data_treat = pd.read_csv('/content/drive/MyDrive/BU for me/Projects at BU/data business analytics /Business experiments and causal data analysis/The Effects of Phone Calls on Voter Turnout/noncompliance_treat_small.csv')
data_placebo = pd.read_csv('/content/drive/MyDrive/BU for me/Projects at BU/data business analytics /Business experiments and causal data analysis/The Effects of Phone Calls on Voter Turnout/noncompliance_placebo.csv')

# Sample sizes
print("Treatment Dataset Sample Size:", data_treat.shape[0])
print("Placebo Dataset Sample Size:", data_placebo.shape[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Treatment Dataset Sample Size: 293412
Placebo Dataset Sample Size: 47540



## 🧠 Part 3: Analysis

We'll estimate the CACE using two methods:

### Method 1: Using Treatment vs Control
1. Estimate ITT using regression.
2. Compute compliance rate (α).
3. Estimate CACE = ITT / α
4. Use `pyfixest` for IV regression (robust CACE).

### Method 2: Using Treatment vs Placebo
1. Estimate placebo compliance effect.
2. Run regressions among contacted individuals.
3. Compare standard errors and assumptions.


In [7]:
import statsmodels.formula.api as smf

# ITT estimation
itt_model = smf.ols('voted_aug2008 ~ treatment_attempt_turnout_call', data=data_treat).fit()
itt = itt_model.params['treatment_attempt_turnout_call']
se_itt = itt_model.bse['treatment_attempt_turnout_call']

print(f"ITT Estimate: {itt:.4f}")
print(f"Standard Error: {se_itt:.4f}")

# Compliance rate
alpha = data_treat[data_treat['treatment_attempt_turnout_call'] == 1]['contacted'].mean()
print(f"Compliance Rate (alpha): {alpha:.4f}")

# CACE estimate
cace_estimate = itt / alpha
cace_se = se_itt / alpha

print(f"CACE (Manual IV Estimate): {cace_estimate:.4f}")
print(f"Adjusted Standard Error: {cace_se:.4f}")


ITT Estimate: 0.0120
Standard Error: 0.0044
Compliance Rate (alpha): 0.5741
CACE (Manual IV Estimate): 0.0209
Adjusted Standard Error: 0.0077


✅ 1. Intent-to-Treat (ITT) Effect
This estimates the effect of being assigned to receive a turnout call, regardless of whether the person was actually contacted.

ITT Estimate = 0.0120

Interpretation: Assignment to receive a turnout call increased the probability of voting by 1.2 percentage points on average.

Statistical Significance: With a standard error of 0.0044, the ITT estimate is statistically significant (t ≈ 2.73).

✅ 2. Compliance Rate (α)
This is the proportion of people in the treatment group who were actually contacted.

Compliance Rate = 0.5741 (≈57.4%)

Interpretation: Not everyone assigned to treatment was contacted, so the treatment was not fully administered.

✅ 3. Complier Average Causal Effect (CACE)
This captures the effect among those who actually complied (i.e., were contacted). Calculated as:

CACE = ITT / Compliance Rate = 0.0120 / 0.5741 ≈ 0.0209

Interpretation: Among those who were actually contacted, voting probability increased by ~2.1 percentage points.

Adjusted SE: 0.0077 — statistically significant.

In [8]:
!pip install pyfixest --quiet

from pyfixest.estimation import feols

# Using pyFixest for IV regression
iv_model = feols("voted_aug2008 ~ 1 | 0 | contacted ~ treatment_attempt_turnout_call", data=data_treat)
print(iv_model.summary())


###

Estimation:  IV
Dep. var.: voted_aug2008, Fixed effects: 0
Inference:  iid
Observations:  293412

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      0.187 |        0.001 |   256.093 |      0.000 |  0.186 |   0.188 |
| contacted     |      0.021 |        0.008 |     2.697 |      0.007 |  0.006 |   0.036 |
---

None


✅ 4. Instrumental Variable (IV) Estimate using pyfixest
The IV estimate is a more formal way to compute CACE using a 2-stage least squares approach:

Estimate = 0.021

SE = 0.008

p-value = 0.007

Interpretation: Consistent with the manual CACE calculation. Contact caused a ~2.1% increase in voting probability among compliers.

In [9]:
# Compare sample sizes
print("Treatment dataset shape:", data_treat.shape)
print("Placebo dataset shape:", data_placebo.shape)

# Regression among placebo group
placebo_model = smf.ols('voted_aug2008 ~ contacted', data=data_placebo[data_placebo['treatment_attempt_turnout_call'] == 0]).fit()
print(placebo_model.summary())


Treatment dataset shape: (293412, 4)
Placebo dataset shape: (47540, 4)
                            OLS Regression Results                            
Dep. Variable:          voted_aug2008   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     74.26
Date:                Mon, 14 Apr 2025   Prob (F-statistic):           7.28e-18
Time:                        02:36:01   Log-Likelihood:                -11266.
No. Observations:               23761   AIC:                         2.254e+04
Df Residuals:                   23759   BIC:                         2.255e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------

✅ 5. Placebo Test
Running the regression in the placebo group (not assigned to treatment):

Estimate = 0.0438

Interpretation: Some "contacted" people were not in the treatment group, and their higher turnout suggests contacting isn't randomly assigned — so a naïve OLS may be biased.

In [10]:
# Restrict to contacted
contacted_only = data_placebo[data_placebo['contacted'] == 1]

# Regression on contacted
cace_contacted_model = smf.ols('voted_aug2008 ~ treatment_attempt_turnout_call', data=contacted_only).fit()
print(cace_contacted_model.summary())


                            OLS Regression Results                            
Dep. Variable:          voted_aug2008   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     36.36
Date:                Mon, 14 Apr 2025   Prob (F-statistic):           1.66e-09
Time:                        02:36:01   Log-Likelihood:                -14427.
No. Observations:               26793   AIC:                         2.886e+04
Df Residuals:                   26791   BIC:                         2.887e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

✅ 6. Restricted to Contacted: CACE Check
Among those actually contacted (in placebo), regressing on assignment:

Effect = 0.0305

Interpretation: Another way to estimate effect among contacted. Slightly higher, but consistent in showing a positive effect.

In [11]:
# Heterogeneous effect test
hetero_model = smf.ols('voted_aug2008 ~ treatment_attempt_turnout_call * voted_nov2002', data=data_treat).fit()
print(hetero_model.summary())


                            OLS Regression Results                            
Dep. Variable:          voted_aug2008   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     1991.
Date:                Mon, 14 Apr 2025   Prob (F-statistic):               0.00
Time:                        02:36:01   Log-Likelihood:            -1.3722e+05
No. Observations:              293412   AIC:                         2.745e+05
Df Residuals:                  293408   BIC:                         2.745e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------


## ✅ Part 4: 💡 Conclusion

This analysis shows that canvassing phone calls have a small but statistically significant causal impact on voter turnout, particularly among those who answer the call. While the overall ITT effect is modest, the true effect among compliers (CACE) is more substantial. The results support the effectiveness of targeted GOTV (Get Out The Vote) strategies, especially for voters with prior engagement history.